# CommercePulse — Análise Exploratória de Dados

**Objetivo:** Realizar a primeira análise exploratória do e-commerce brasileiro, com foco em KPIs, receita, localização, categorias, logística e satisfação do cliente.

**Base de dados:** `data/processed/commercepulse_orders_items.csv` — granularidade: **1 linha por item vendido**.

**Perguntas de negócio:**
1. Qual foi a evolução da receita ao longo do tempo?
2. Quais estados mais compram?
3. Quais categorias mais geram receita?
4. Qual é o ticket médio?
5. Qual é a taxa de atraso?
6. Atraso afeta avaliação?
7. Frete alto afeta avaliação?
8. Quais categorias têm pior satisfação?
9. Quais estados têm maior atraso?
10. Quais oportunidades de melhoria aparecem?

---
## 1. Imports e Configurações

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Template visual padrão dos gráficos
PLOTLY_TEMPLATE = "plotly_dark"
COLOR_PALETTE = px.colors.qualitative.Set2

Matplotlib is building the font cache; this may take a moment.


---
## 2. Carregamento da Base Processada

In [3]:
df = pd.read_csv(PROCESSED_DIR / "commercepulse_orders_items.csv")

# Converter colunas de data para datetime
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date", "purchase_date"
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Base carregada: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

Base carregada: 112,650 linhas x 44 colunas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,total_payment_value,review_score,purchase_date,purchase_year,purchase_month,purchase_year_month,delivery_days,estimated_delivery_days,delivery_delta_days,delay_days,is_late,is_delivered,item_revenue,item_total_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.00,268.00,4.00,500.00,19.00,8.00,13.00,housewares,9350,maua,SP,38.71,4.00,2017-10-02,2017,10,2017-10,8.00,15,-8.00,0.00,0.00,1,29.99,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.00,178.00,1.00,400.00,19.00,13.00,19.00,perfumery,31570,belo horizonte,SP,141.46,4.00,2018-07-24,2018,7,2018-07,13.00,19,-6.00,0.00,0.00,1,118.70,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.00,232.00,1.00,420.00,24.00,19.00,21.00,auto,14840,guariba,SP,179.12,5.00,2018-08-08,2018,8,2018-08,9.00,26,-18.00,0.00,0.00,1,159.90,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.00,468.00,3.00,450.00,30.00,10.00,20.00,pet_shop,31842,belo horizonte,MG,72.20,5.00,2017-11-18,2017,11,2017-11,13.00,26,-13.00,0.00,0.00,1,45.00,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.00,316.00,4.00,250.00,51.00,15.00,15.00,stationery,8752,mogi das cruzes,SP,28.62,5.00,2018-02-13,2018,2,2018-02,2.00,12,-10.00,0.00,0.00,1,19.90,28.62


---
## 3. Validação Inicial

Antes de qualquer análise, verificamos a integridade dos dados: tipos, valores ausentes e contagens de entidades principais.

In [4]:
print("=" * 60)
print("ESTRUTURA DA BASE")
print("=" * 60)
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")
print(f"Pedidos únicos: {df['order_id'].nunique():,}")
print(f"Clientes únicos: {df['customer_unique_id'].nunique():,}")
print(f"Vendedores únicos: {df['seller_id'].nunique():,}")
print(f"Categorias únicas: {df['product_category_name_english'].nunique()}")
print(f"Estados atendidos: {df['customer_state'].nunique()}")
print("=" * 60)

ESTRUTURA DA BASE
Linhas: 112,650
Colunas: 44
Pedidos únicos: 98,666
Clientes únicos: 95,420
Vendedores únicos: 3,095
Categorias únicas: 71
Estados atendidos: 27


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 44 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  str           
 1   customer_id                    112650 non-null  str           
 2   order_status                   112650 non-null  str           
 3   order_purchase_timestamp       112650 non-null  datetime64[us]
 4   order_approved_at              112635 non-null  datetime64[us]
 5   order_delivered_carrier_date   111456 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  112650 non-null  datetime64[us]
 8   customer_unique_id             112650 non-null  str           
 9   customer_zip_code_prefix       112650 non-null  int64         
 10  customer_city                  112650 non-null  str           
 11  customer_st

In [6]:
# Percentual de valores ausentes por coluna
missing = df.isna().mean().sort_values(ascending=False).head(20)
missing_df = pd.DataFrame({"coluna": missing.index, "% ausente": (missing.values * 100).round(2)})
missing_df[missing_df["% ausente"] > 0]

,coluna,% ausente
0,order_delivered_customer_date,2.18
1,delay_days,2.18
2,delivery_delta_days,2.18
3,delivery_days,2.18
4,is_late,2.18
5,product_category_name_english,1.44
6,product_category_name,1.42
7,product_description_lenght,1.42
8,product_name_lenght,1.42
9,product_photos_qty,1.42


---
## 4. KPIs Principais

Visão geral das métricas que resumem o desempenho do e-commerce.

> **Nota:** Usamos `item_revenue` (= price) para receita, e **não** `total_payment_value`, pois este último está no nível do pedido e seria duplicado se somado na base de itens.

In [7]:
# Ticket médio calculado no nível do pedido
orders_revenue = df.groupby("order_id")["item_revenue"].sum().reset_index()

kpis = {
    "Total de pedidos": f"{df['order_id'].nunique():,}",
    "Total de itens vendidos": f"{len(df):,}",
    "Receita total (itens)": f"R$ {df['item_revenue'].sum():,.2f}",
    "Receita total (itens + frete)": f"R$ {df['item_total_value'].sum():,.2f}",
    "Ticket médio por pedido": f"R$ {orders_revenue['item_revenue'].mean():,.2f}",
    "Nota média dos clientes": f"{df['review_score'].mean():.2f}",
    "Tempo médio de entrega (dias)": f"{df['delivery_days'].mean():.1f}",
    "Taxa de atraso": f"{df['is_late'].mean() * 100:.1f}%",
    "Categorias de produto": f"{df['product_category_name_english'].nunique()}",
    "Estados atendidos": f"{df['customer_state'].nunique()}",
}

kpis_df = pd.DataFrame(kpis.items(), columns=["Métrica", "Valor"])
kpis_df

,Métrica,Valor
0,Total de pedidos,"98,666"
1,Total de itens vendidos,"112,650"
2,Receita total (itens),"R$ 13,591,643.70"
3,Receita total (itens + frete),"R$ 15,843,553.24"
4,Ticket médio por pedido,R$ 137.75
5,Nota média dos clientes,4.03
6,Tempo médio de entrega (dias),12.0
7,Taxa de atraso,7.9%
8,Categorias de produto,71
9,Estados atendidos,27


---
## 5. Tabelas Agregadas

Criamos as principais tabelas que serão reutilizadas nas análises e gráficos.

In [8]:
# --- monthly_revenue ---
monthly_revenue = (
    df.groupby("purchase_year_month")
    .agg(
        revenue=("item_revenue", "sum"),
        total_value=("item_total_value", "sum"),
        orders=("order_id", "nunique"),
        items=("order_item_id", "count"),
        avg_review=("review_score", "mean"),
    )
    .reset_index()
)
monthly_revenue["ticket_medio"] = monthly_revenue["revenue"] / monthly_revenue["orders"]

print("=== Receita Mensal ===")
monthly_revenue.tail(10)

=== Receita Mensal ===


,purchase_year_month,revenue,total_value,orders,items,avg_review,ticket_medio
14,2017-12,"743,914.17","863,547.23",5624,6308,3.96,132.27
15,2018-01,"950,030.36","1,107,301.89",7220,8208,3.95,131.58
16,2018-02,"844,178.71","986,908.96",6694,7672,3.74,126.11
17,2018-03,"983,213.44","1,155,126.82",7188,8217,3.70,136.79
18,2018-04,"996,647.75","1,159,698.04",6934,7975,4.08,143.73
19,2018-05,"996,517.68","1,149,781.82",6853,7925,4.13,145.41
20,2018-06,"865,124.31","1,022,677.11",6160,7078,4.19,140.44
21,2018-07,"895,507.22","1,058,728.03",6273,7092,4.23,142.76
22,2018-08,"854,686.33","1,003,308.47",6452,7248,4.22,132.47
23,2018-09,145.00,166.46,1,1,1.00,145.00


In [9]:
# --- state_revenue ---
state_revenue = (
    df.groupby("customer_state")
    .agg(
        revenue=("item_revenue", "sum"),
        total_value=("item_total_value", "sum"),
        orders=("order_id", "nunique"),
        items=("order_item_id", "count"),
        avg_review=("review_score", "mean"),
        avg_freight=("freight_value", "mean"),
        late_rate=("is_late", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)
state_revenue["ticket_medio"] = state_revenue["revenue"] / state_revenue["orders"]

print("=== Top 10 Estados por Receita ===")
state_revenue.head(10)

=== Top 10 Estados por Receita ===


,customer_state,revenue,total_value,orders,items,avg_review,avg_freight,late_rate,avg_delivery_days,ticket_medio
25,SP,"5,202,955.05","5,921,678.12",41375,47449,4.13,15.15,0.06,8.26,125.75
18,RJ,"1,824,092.67","2,129,681.98",12762,14579,3.81,20.96,0.13,14.69,142.93
10,MG,"1,585,308.03","1,856,161.49",11544,13129,4.09,20.63,0.05,11.52,137.33
22,RS,"750,304.02","885,826.76",5432,6235,4.05,21.74,0.07,14.71,138.13
17,PR,"683,083.76","800,935.44",4998,5740,4.11,20.53,0.05,11.48,136.67
23,SC,"520,553.34","610,213.60",3612,4176,4.00,21.47,0.10,14.52,144.12
4,BA,"511,349.99","611,506.67",3358,3799,3.81,26.36,0.14,18.77,152.28
6,DF,"302,603.94","353,229.44",2125,2406,4.01,21.04,0.07,12.50,142.40
8,GO,"294,591.95","347,706.93",2007,2333,3.99,22.77,0.08,14.95,146.78
7,ES,"275,037.31","324,801.91",2025,2256,3.99,22.06,0.12,15.19,135.82


In [10]:
# --- category_revenue ---
category_revenue = (
    df.groupby("product_category_name_english")
    .agg(
        revenue=("item_revenue", "sum"),
        orders=("order_id", "nunique"),
        items=("order_item_id", "count"),
        avg_review=("review_score", "mean"),
        avg_freight=("freight_value", "mean"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

print("=== Top 10 Categorias por Receita ===")
category_revenue.head(10)

=== Top 10 Categorias por Receita ===


,product_category_name_english,revenue,orders,items,avg_review,avg_freight,late_rate
43,health_beauty,"1,258,681.34",8836,9670,4.14,18.88,0.09
70,watches_gifts,"1,205,005.68",5624,5991,4.02,16.78,0.08
7,bed_bath_table,"1,036,988.68",9417,11115,3.90,18.42,0.08
65,sports_leisure,"988,048.97",7720,8641,4.11,19.51,0.07
15,computers_accessories,"911,954.32",6689,7827,3.93,18.82,0.08
39,furniture_decor,"729,762.49",6449,8334,3.91,20.73,0.08
20,cool_stuff,"635,290.85",3632,3796,4.15,22.14,0.07
49,housewares,"632,248.66",5884,6964,4.05,20.99,0.06
5,auto,"592,720.11",3897,4235,4.06,21.88,0.08
42,garden_tools,"485,256.46",3518,4347,4.05,22.77,0.08


In [11]:
# --- delivery_analysis ---
delivered = df[df["is_delivered"] == 1].copy()

delivery_analysis = (
    delivered.groupby("customer_state")
    .agg(
        avg_delivery_days=("delivery_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        avg_delay=("delay_days", "mean"),
        avg_delta=("delivery_delta_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review=("review_score", "mean"),
        orders=("order_id", "nunique"),
    )
    .reset_index()
    .sort_values("avg_delivery_days", ascending=False)
)

print("=== Análise de Entrega por Estado ===")
delivery_analysis.head(10)

=== Análise de Entrega por Estado ===


,customer_state,avg_delivery_days,median_delivery_days,avg_delay,avg_delta,late_rate,avg_review,orders
21,RR,27.83,24.50,3.96,-18.33,0.11,3.89,41
3,AP,27.75,24.00,3.57,-18.40,0.05,4.26,67
2,AM,25.96,26.00,0.75,-19.93,0.04,4.11,145
1,AL,23.99,22.00,2.04,-8.74,0.24,3.83,397
13,PA,23.30,21.00,1.41,-14.25,0.12,3.84,946
9,MA,21.20,19.00,1.78,-9.91,0.20,3.76,717
24,SE,20.98,17.00,2.37,-10.00,0.16,3.90,335
5,CE,20.54,18.00,2.00,-11.10,0.15,3.87,1279
0,AC,20.33,18.00,0.62,-20.98,0.03,4.13,80
14,PB,20.12,18.00,1.07,-13.04,0.11,4.04,517


In [12]:
# --- review_by_delay ---
review_by_delay = (
    delivered.groupby("is_late")
    .agg(
        avg_review=("review_score", "mean"),
        median_review=("review_score", "median"),
        count=("order_id", "count"),
        avg_delay_days=("delay_days", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
    )
    .reset_index()
)
review_by_delay["is_late"] = review_by_delay["is_late"].map({0: "No prazo", 1: "Atrasado"})

print("=== Avaliação por Status de Entrega ===")
review_by_delay

=== Avaliação por Status de Entrega ===


,is_late,avg_review,median_review,count,avg_delay_days,avg_delivery_days
0,No prazo,4.21,5.00,101475,0.00,10.39
1,Atrasado,2.55,2.00,8714,8.74,30.89


---
## 6. Análises e Gráficos

### 6.1 Evolução da Receita Mensal
**Pergunta:** Qual foi a evolução da receita ao longo do tempo?

In [13]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Receita Mensal (R$)", "Pedidos por Mês"),
    vertical_spacing=0.12,
)

fig.add_trace(
    go.Scatter(
        x=monthly_revenue["purchase_year_month"],
        y=monthly_revenue["revenue"],
        mode="lines+markers",
        name="Receita",
        line=dict(color="#00d2ff", width=2.5),
        marker=dict(size=5),
        fill="tozeroy",
        fillcolor="rgba(0,210,255,0.1)",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=monthly_revenue["purchase_year_month"],
        y=monthly_revenue["orders"],
        name="Pedidos",
        marker_color="#7c4dff",
    ),
    row=2, col=1,
)

fig.update_layout(
    title="Evolução da Receita e Volume de Pedidos",
    template=PLOTLY_TEMPLATE,
    height=600,
    showlegend=False,
    title_font_size=18,
)
fig.update_yaxes(title_text="Receita (R$)", row=1, col=1)
fig.update_yaxes(title_text="Pedidos", row=2, col=1)
fig.update_xaxes(title_text="Mês", row=2, col=1)

fig.show()

### 6.2 Top 10 Estados por Receita
**Pergunta:** Quais estados mais compram?

In [14]:
top_states = state_revenue.head(10).sort_values("revenue")

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=top_states["customer_state"],
        x=top_states["revenue"],
        orientation="h",
        marker=dict(
            color=top_states["revenue"],
            colorscale="Tealgrn",
        ),
        text=top_states["revenue"].apply(lambda x: f"R$ {x:,.0f}"),
        textposition="outside",
    )
)

fig.update_layout(
    title="Top 10 Estados por Receita",
    xaxis_title="Receita (R$)",
    yaxis_title="Estado",
    template=PLOTLY_TEMPLATE,
    height=500,
    title_font_size=18,
)

fig.show()

### 6.3 Top 10 Categorias por Receita
**Pergunta:** Quais categorias mais geram receita?

In [15]:
top_categories = category_revenue.head(10).sort_values("revenue")

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=top_categories["product_category_name_english"],
        x=top_categories["revenue"],
        orientation="h",
        marker=dict(
            color=top_categories["revenue"],
            colorscale="Sunset",
        ),
        text=top_categories["revenue"].apply(lambda x: f"R$ {x:,.0f}"),
        textposition="outside",
    )
)

fig.update_layout(
    title="Top 10 Categorias por Receita",
    xaxis_title="Receita (R$)",
    yaxis_title="Categoria",
    template=PLOTLY_TEMPLATE,
    height=500,
    title_font_size=18,
)

fig.show()

### 6.4 Relação entre Atraso e Avaliação
**Pergunta:** Atraso afeta avaliação?

In [16]:
fig = go.Figure()

colors = {"No prazo": "#00e676", "Atrasado": "#ff5252"}

for _, row in review_by_delay.iterrows():
    fig.add_trace(
        go.Bar(
            x=[row["is_late"]],
            y=[row["avg_review"]],
            name=row["is_late"],
            marker_color=colors.get(row["is_late"], "#888"),
            text=f"{row['avg_review']:.2f}",
            textposition="outside",
            width=0.5,
        )
    )

fig.update_layout(
    title="Nota Média: Pedidos no Prazo vs. Atrasados",
    yaxis_title="Nota Média",
    yaxis_range=[0, 5.5],
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False,
    title_font_size=18,
)

fig.show()

### 6.5 Frete Médio vs. Avaliação Média por Estado
**Pergunta:** Estados com frete mais caro têm pior avaliação?

In [17]:
fig = px.scatter(
    state_revenue,
    x="avg_freight",
    y="avg_review",
    size="orders",
    color="late_rate",
    text="customer_state",
    color_continuous_scale="RdYlGn_r",
    labels={
        "avg_freight": "Frete Médio (R$)",
        "avg_review": "Nota Média",
        "orders": "Pedidos",
        "late_rate": "Taxa de Atraso",
    },
    title="Frete Médio vs. Avaliação Média por Estado",
    template=PLOTLY_TEMPLATE,
    height=550,
)

fig.update_traces(textposition="top center", textfont_size=9)
fig.update_layout(title_font_size=18)

fig.show()

### 6.6 Estados com Maior Taxa de Atraso
**Pergunta:** Quais estados têm maior atraso?

In [18]:
# Filtramos estados com pelo menos 100 pedidos para evitar distorções
states_filtered = delivery_analysis[delivery_analysis["orders"] >= 100].sort_values("late_rate", ascending=False).head(15)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=states_filtered["customer_state"],
        y=states_filtered["late_rate"] * 100,
        marker=dict(
            color=states_filtered["late_rate"],
            colorscale="OrRd",
        ),
        text=states_filtered["late_rate"].apply(lambda x: f"{x*100:.1f}%"),
        textposition="outside",
    )
)

fig.update_layout(
    title="Taxa de Atraso por Estado (mín. 100 pedidos)",
    xaxis_title="Estado",
    yaxis_title="Taxa de Atraso (%)",
    template=PLOTLY_TEMPLATE,
    height=500,
    title_font_size=18,
)

fig.show()

### 6.7 Categorias com Melhor e Pior Satisfação
**Pergunta:** Quais categorias têm melhor e pior satisfação?

In [19]:
# Filtramos categorias com pelo menos 50 avaliações
cat_filtered = category_revenue[category_revenue["items"] >= 50].copy()

top5_best = cat_filtered.nlargest(5, "avg_review")
top5_worst = cat_filtered.nsmallest(5, "avg_review")
cat_extremes = pd.concat([top5_best, top5_worst]).sort_values("avg_review")

colors = ["#ff5252"] * len(top5_worst) + ["#00e676"] * len(top5_best)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=cat_extremes["product_category_name_english"],
        x=cat_extremes["avg_review"],
        orientation="h",
        marker_color=colors,
        text=cat_extremes["avg_review"].apply(lambda x: f"{x:.2f}"),
        textposition="outside",
    )
)

fig.update_layout(
    title="Top 5 Melhores e Piores Categorias por Avaliação",
    xaxis_title="Nota Média",
    xaxis_range=[0, 5.5],
    yaxis_title="Categoria",
    template=PLOTLY_TEMPLATE,
    height=550,
    title_font_size=18,
)

fig.show()

### 6.8 Ticket Médio por Estado
**Pergunta:** Qual é o ticket médio por estado?

In [20]:
top_ticket = state_revenue.sort_values("ticket_medio", ascending=False).head(15)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_ticket["customer_state"],
        y=top_ticket["ticket_medio"],
        marker=dict(
            color=top_ticket["ticket_medio"],
            colorscale="Viridis",
        ),
        text=top_ticket["ticket_medio"].apply(lambda x: f"R$ {x:,.0f}"),
        textposition="outside",
    )
)

fig.update_layout(
    title="Ticket Médio por Estado (Top 15)",
    xaxis_title="Estado",
    yaxis_title="Ticket Médio (R$)",
    template=PLOTLY_TEMPLATE,
    height=500,
    title_font_size=18,
)

fig.show()

---
## 7. Conclusões Iniciais

### 📊 Insights

**Insight 1 — Atraso impacta diretamente a satisfação do cliente.**  
Pedidos entregues com atraso apresentam avaliação média significativamente inferior aos pedidos entregues no prazo, indicando que logística é um fator crítico para a satisfação do cliente.

**Insight 2 — Concentração geográfica da receita.**  
Os estados de SP, RJ e MG concentram a maior parte da receita e do volume de pedidos, refletindo a concentração econômica e populacional do Sudeste brasileiro.

**Insight 3 — Frete elevado está correlacionado com pior avaliação.**  
Estados mais distantes dos grandes centros logísticos pagam frete mais caro e tendem a apresentar notas médias mais baixas, o que sugere uma oportunidade de otimização logística.

**Insight 4 — Categorias de alto valor têm comportamento distinto.**  
Categorias como *computers* e *watches_gifts* geram receita relevante com menos itens vendidos, indicando alto ticket médio. Já categorias como *bed_bath_table* dependem de volume.

**Insight 5 — Crescimento consistente até pico seguido de estabilização.**  
A receita mensal mostra crescimento consistente ao longo do período analisado, com picos sazonais (ex.: Black Friday). Isso indica um mercado em expansão, mas com oportunidades de suavizar a sazonalidade.

---

### 💡 Recomendações Executivas

**Recomendação 1 — Investir em logística para reduzir atrasos.**  
A correlação entre atraso e nota baixa é clara. Reduzir o tempo de entrega, especialmente em estados do Norte e Nordeste, pode melhorar significativamente a satisfação e reduzir churn.

**Recomendação 2 — Criar estratégia de frete subsidiado para estados de alto potencial.**  
Estados com alto frete e boa base de clientes podem ser incentivados com programas de frete grátis ou reduzido, aumentando conversão e recompra.

**Recomendação 3 — Monitorar categorias com baixa satisfação.**  
Categorias com avaliação consistentemente baixa devem ser investigadas para problemas de qualidade de produto, descrição enganosa ou expectativa não atendida. Ações corretivas podem incluir curadoria de vendedores e melhoria de descrições.

---

### Próximos passos

- Aprofundar análise de vendedores com maior concentração de atrasos
- Construir dashboard interativo consolidando as métricas e gráficos
- Criar relatório executivo com storytelling dos insights
- Segmentar clientes por comportamento de compra (RFM)